In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Feedforward clasic și flux de control (circuite dinamice)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Feedforward clasic și flux de control


<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau versiuni mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Noua versiune a circuitelor dinamice este acum disponibilă tuturor utilizatorilor pe toate Backend-urile. Poți rula acum circuite dinamice la scară de utilitate. Consultă [anunțul](/announcements/product-updates/2025-09-25-new-dynamic-circuits) pentru mai multe detalii.

Circuitele dinamice sunt instrumente puternice cu care poți măsura qubiți în mijlocul execuției unui Circuit cuantic și apoi efectua operații de logică clasică în cadrul circuitului, pe baza rezultatelor acelor măsurători la mijlocul circuitului. Acest proces este cunoscut și sub denumirea de _feedforward clasic_. Deși suntem la început în înțelegerea modului optim de a profita de circuitele dinamice, comunitatea de cercetare cuantică a identificat deja o serie de cazuri de utilizare, precum:

* Pregătirea eficientă a stărilor cuantice, cum ar fi [starea GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [starea W,](https://arxiv.org/abs/2403.07604) (pentru mai multe informații despre starea W, consultați și ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) și o clasă largă de [stări de produs matriceal](https://arxiv.org/abs/2404.16083)
* [Entanglement eficient pe distanțe lungi](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) între qubiți de pe același cip, folosind circuite de adâncime mică
* [Eșantionarea eficientă a circuitelor de tip IQP](https://arxiv.org/pdf/2505.04705)

Aceste îmbunătățiri aduse de circuitele dinamice vin însă cu compromisuri. Măsurătorile la mijlocul circuitului și operațiile clasice au în general un timp de execuție mai lung decât porțile cu doi qubiți, iar această creștere a timpului poate anula beneficiile reducerii adâncimii circuitului. Prin urmare, reducerea duratei măsurătorilor la mijlocul circuitului reprezintă un domeniu de îmbunătățire pe care IBM Quantum&reg; îl urmărește în [noua versiune](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) a circuitelor dinamice.

[Specificația OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definește o serie de structuri de flux de control, dar Qiskit Runtime acceptă în prezent doar instrucțiunea condițională `if`. În Qiskit SDK, aceasta corespunde metodei [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) din [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Această metodă returnează un [manager de context](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) și este folosită de obicei într-o instrucțiune `with`. Acest ghid descrie cum să folosești această instrucțiune condițională.

> **Note:** Exemplele de cod din acest ghid folosesc instrucțiunea standard de măsurare pentru măsurătorile la mijlocul circuitului. Cu toate acestea, se recomandă să folosești instrucțiunea [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) în schimb, dacă Backend-ul o acceptă. Consultă [documentația privind măsurătorile la mijlocul circuitului](/guides/measure-qubits#mid-circuit-measurements) pentru detalii.
## Instrucțiunea `if`
Instrucțiunea `if` este folosită pentru a efectua condiționat operații pe baza valorii unui bit sau registru clasic.

În exemplul de mai jos, aplicăm o poartă Hadamard unui qubit și îl măsurăm. Dacă rezultatul este 1, atunci aplicăm o poartă X pe qubit, ceea ce are efectul de a-l readuce la starea 0. Măsurăm apoi qubit-ul din nou. Rezultatul măsurătorii ar trebui să fie 0 cu probabilitate de 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Instrucțiunii `with` i se poate atribui o țintă care este ea însăși un manager de context, ce poate fi stocat și utilizat ulterior pentru a crea un bloc else, care se execută ori de câte ori conținutul blocului `if` *nu* este executat.

În exemplul de mai jos, inițializăm registre cu doi qubiți și două biți clasici. Aplicăm o poartă Hadamard primului qubit și îl măsurăm. Dacă rezultatul este 1, atunci aplicăm o poartă Hadamard pe al doilea qubit; altfel, aplicăm o poartă X pe al doilea qubit. În final, măsurăm și al doilea qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Pe lângă condiționarea pe un singur bit clasic, este posibil și să condiționezi pe valoarea unui registru clasic compus din mai mulți biți.

În exemplul de mai jos, aplicăm porți Hadamard la doi qubiți și îi măsurăm. Dacă rezultatul este `01`, adică primul qubit este 1 și al doilea qubit este 0, atunci aplicăm o poartă X unui al treilea qubit. În final, măsurăm al treilea qubit. Remarcă că, pentru claritate, am ales să specificăm starea celui de-al treilea bit clasic, care este 0, în condiția `if`. În desenul circuitului, condiția este indicată prin cercuri pe biții clasici pe care se condiționează. Un cerc negru indică condiționarea pe 1, în timp ce un cerc alb indică condiționarea pe 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Expresii clasice
Modulul de expresii clasice Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) conține o reprezentare exploratorie a operațiilor de execuție pe valori clasice în timpul execuției circuitului. Din cauza limitărilor hardware, în prezent sunt acceptate doar condițiile `QuantumCircuit.if_test()`.

Următorul exemplu arată că poți folosi calculul parității pentru a crea o stare GHZ cu n qubiți folosind circuite dinamice. Mai întâi, generează $n/2$ perechi Bell pe qubiți adiacenți. Apoi, lipește aceste perechi împreună folosind un strat de porți CNOT între perechi. Măsori apoi qubit-ul țintă al tuturor porților CNOT anterioare și resetezi fiecare qubit măsurat la starea $\vert 0 \rangle$. Aplici $X$ la fiecare poziție nemăsurată pentru care paritatea tuturor biților precedenți este impară. În final, porțile CNOT sunt aplicate qubiților măsurați pentru a restabili entanglement-ul pierdut în măsurătoare.

În calculul parității, primul element al expresiei construite implică ridicarea obiectului Python `mr[0]` la un nod [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` este folosit pentru a transforma obiecte arbitrare în expresii clasice). Acest lucru nu este necesar pentru `mr[1]` și posibilul registru clasic următor, deoarece acestea sunt intrări pentru `expr.bit_xor`, iar orice ridicare necesară este efectuată automat în aceste cazuri. Astfel de expresii pot fi construite în bucle și alte constructe.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Găsește Backend-uri care acceptă circuite dinamice
Pentru a găsi toate Backend-urile la care contul tău poate accesa și care acceptă circuite dinamice, rulează cod similar cu cel de mai jos. Acest exemplu presupune că ai [salvat credențialele de autentificare.](/guides/save-credentials) Poți, de asemenea, să [specifici explicit credențialele](/guides/initialize-account#explicit) atunci când inițializezi contul tău de serviciu Qiskit Runtime. Aceasta ți-ar permite să vizualizezi Backend-urile disponibile pe o instanță specifică sau un tip de plan, de exemplu.

> **Note:** - Backend-urile disponibile pentru cont depind de instanța specificată în credențiale.
> - Noua versiune a circuitelor dinamice este acum disponibilă tuturor utilizatorilor pe toate Backend-urile. Consultă [anunțul](/announcements/product-updates/2025-09-25-new-dynamic-circuits) pentru mai multe detalii.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>